# 1. Setup dan Integrasi Preprocessing

In [ ]:
import pandas as pd
import numpy as np
import string
import re
from sentence_transformers import SentenceTransformer, util
from tqdm.auto import tqdm
import torch

# Load Data
QUERY_PATH = 'data/processed/queries_synthetic_v2.csv'
TAFSIR_PATH = 'data/processed/tafsir_clean.csv'

df_queries = pd.read_csv(QUERY_PATH)
df_tafsir = pd.read_csv(TAFSIR_PATH)

# Fungsi Preprocessing 
def clean_for_mining(text):
    if pd.isna(text): return ""
    text = str(text).lower()
    # Hapus tanda baca
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Hapus karakter non-alfabet
    text = re.sub(r'[^a-z\s]', '', text)
    return " ".join(text.split())

print("Melakukan Preprocessing pada kueri dan korpus...")
df_queries['clean_query'] = df_queries['query'].apply(clean_for_mining)
df_tafsir['clean_tafsir'] = df_tafsir['tafsir_text'].apply(clean_for_mining)

# 2. Encoding dan Semantic Mining

In [ ]:
# Load Fine-Tuned Model
model_path = 'models/sbert_tafsir_finetuned' 
model = SentenceTransformer(model_path, device='cuda' if torch.cuda.is_available() else 'cpu')

corpus_texts = df_tafsir['clean_tafsir'].tolist()
raw_tafsir = df_tafsir['tafsir_text'].tolist() # Untuk output teks asli

print("Encoding seluruh korpus tafsir...")
corpus_embeddings = model.encode(corpus_texts, convert_to_tensor=True, show_progress_bar=True)

# 3. Eksekusi Hard Negative Mining (Rasio 1:3)

In [ ]:
SKIP_N = 5     # Lewati 5 hasil teratas
TAKE_N = 3     # Ambil 3 dokumen untuk rasio 1:3

triplets = []

print(f"⛏️ Mining {TAKE_N} Hard Negatives per kueri...")

for idx, row in tqdm(df_queries.iterrows(), total=len(df_queries)):
    query_text = row['clean_query']
    pos_text = row['clean_tafsir']
    
    # Encode kueri
    query_vec = model.encode(query_text, convert_to_tensor=True)
    
    # Semantic Search
    hits = util.semantic_search(query_vec, corpus_embeddings, top_k=SKIP_N + TAKE_N + 1)[0]
    
    # Filter dan Ambil Negatif
    neg_count = 0
    for hit in hits:
        if corpus_texts[hit['corpus_id']] != pos_text and hit['corpus_id'] > SKIP_N:
            triplets.append({'query': row['query'],           
                'tafsir_text': raw_tafsir[hit['corpus_id']],
                'label': 0})
            neg_count += 1
            if neg_count == TAKE_N: break
    
    triplets.append({
        'query': row['query'],
        'tafsir_text': row['tafsir_text'],
        'label': 1
    })

df_train_final = pd.DataFrame(triplets)
print(f"Total baris dataset: {len(df_train_final)}")

# 4. Export Dataset Pelatihan

In [ ]:
df_train_final = df_train_final.sample(frac=1).reset_index(drop=True)

# Simpan
df_train_final.to_csv('data/processed/dataset_training_FULL.csv', index=False)
print("File 'dataset_training_FULL.csv' siap digunakan.")